<a href="https://colab.research.google.com/github/k2herat/MMK/blob/HW3/Almetov_HW3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!apt-get update
!apt-get install -y aria2
!aria2c -x 16 -s 16 "https://rndml-team-cv.obs.ru-moscow-1.hc.sbercloud.ru/datasets/bukva/bukva.zip" -d -o bukva.zip

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,473 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Pac

In [2]:
!wget -O bukva.zip "https://rndml-team-cv.obs.ru-moscow-1.hc.sbercloud.ru/datasets/bukva/bukva.zip"


--2026-03-30 18:14:38--  https://rndml-team-cv.obs.ru-moscow-1.hc.sbercloud.ru/datasets/bukva/bukva.zip
Resolving rndml-team-cv.obs.ru-moscow-1.hc.sbercloud.ru (rndml-team-cv.obs.ru-moscow-1.hc.sbercloud.ru)... 46.243.206.34, 46.243.206.35
Connecting to rndml-team-cv.obs.ru-moscow-1.hc.sbercloud.ru (rndml-team-cv.obs.ru-moscow-1.hc.sbercloud.ru)|46.243.206.34|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 27077262907 (25G) [application/zip]
Saving to: ‘bukva.zip’

bukva.zip           100%[===================>]  25.22G  4.61MB/s    in 98m 21s 

2026-03-30 19:53:02 (4.38 MB/s) - ‘bukva.zip’ saved [27077262907/27077262907]



In [3]:
!unzip bukva.zip -d burka/


Выходные данные были обрезаны до нескольких последних строк (5000).
  inflating: burka/original/b31f3f48-3b3e-4938-bd7c-a294686bbc92.mp4  
  inflating: burka/original/b3366ab6-7529-4758-93ca-f0eb3e59e977.mp4  
  inflating: burka/original/b3480f8e-6686-4303-8173-380594802fa3.mp4  
  inflating: burka/original/b35d66c1-c5cb-4167-abc0-b580fa5034a3.mp4  
  inflating: burka/original/b36b6a9f-1cf1-40c7-9c22-32567ed3b506.mp4  
  inflating: burka/original/b3901dcd-48fc-412d-b0fd-a6c3aaaff131.mp4  
  inflating: burka/original/b3a21a41bd259a721b29d81d5aa9b177.mp4  
  inflating: burka/original/b3bbd744d20662680502c613fe112f3a.mp4  
  inflating: burka/original/b3c22fb4-50e2-46db-9a24-2a90130118ae.mp4  
  inflating: burka/original/b3d74c62-001b-4fc5-b98d-441930dda67a.mp4  
  inflating: burka/original/b3d8b152-9f52-44a4-870d-d1a573d094b9.mp4  
  inflating: burka/original/b3df3ea7-d2b9-42fb-9f81-99c852bf47e3.mp4  
  inflating: burka/original/b3e63be47c94d13a21ca067a2d8c8b1c.mp4  
  inflating: burka/or

In [25]:
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

ROOT_DIR = "/content/burka/trimmed"
ANNOT_PATH = "/content/burka/annotations.tsv"
CACHE_DIR = "frames_cache"

FRAME_COUNT = 16
IMG_SIZE = 224

In [26]:
meta = pd.read_csv(ANNOT_PATH, sep="\t")

meta["path"] = meta["attachment_id"].apply(
    lambda x: os.path.join(ROOT_DIR, f"{x}.mp4")
)

CLASSES = {
    0: "no_event",
    1: "Ё", 2: "А", 3: "Б", 4: "В",
    5: "Г", 6: "Д", 7: "Е", 8: "Ж", 9: "З"
}

meta = meta[meta["text"].isin(CLASSES.values())].copy()

train_df = meta[meta["train"] == True].copy()
test_df = meta[meta["train"] == False].copy()

train_df, val_df = train_test_split(
    train_df,
    test_size=0.1,
    random_state=42,
    stratify=train_df["text"]
)

print(len(train_df), len(val_df), len(test_df))

837 93 200


In [27]:
import cv2
from tqdm import tqdm

def preprocess_split(df, split):
    save_dir = os.path.join(CACHE_DIR, split)
    os.makedirs(save_dir, exist_ok=True)

    for _, row in tqdm(df.iterrows(), total=len(df), desc=split):
        save_path = os.path.join(save_dir, f"{row['attachment_id']}.npy")

        cap = cv2.VideoCapture(row["path"])
        frame_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        if frame_total <= 0:
            dummy = np.zeros((FRAME_COUNT, IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
            np.save(save_path, dummy)
            cap.release()
            continue

        indices = np.linspace(0, frame_total - 1, FRAME_COUNT).astype(int)

        buffer = []
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ok, frame = cap.read()

            if ok:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frame = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
            else:
                frame = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)

            buffer.append(frame)

        cap.release()
        np.save(save_path, np.array(buffer, dtype=np.uint8))

In [28]:
preprocess_split(train_df, "train")
preprocess_split(val_df, "val")
preprocess_split(test_df, "test")

test: 100%|██████████| 200/200 [11:21<00:00,  3.41s/it]


In [29]:
import torch
from torch.utils.data import Dataset
import torchvision.transforms as transforms

def augment_video(video, size=224, frames=16):
    video = torch.tensor(video, dtype=torch.float32).permute(0,3,1,2) / 255.0

    aug = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomResizedCrop(size, scale=(0.8, 1.0)),
        transforms.ColorJitter(0.2, 0.2, 0.2)
    ])

    video = torch.stack([aug(frame) for frame in video])

    if video.shape[0] > frames:
        idx = torch.randperm(video.shape[0])[:frames]
        video = video[idx]

    mean = torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1)

    return (video - mean) / std


class GestureDataset(Dataset):
    def __init__(self, df, folder, transform=None):
        self.data = df.reset_index(drop=True)
        self.folder = folder
        self.transform = transform

        self.labels = sorted(df["text"].unique())
        self.label_map = {v: i for i, v in enumerate(self.labels)}

    def __len__(self):
        return len(self.data)

    def __getitem__(self, i):
        row = self.data.iloc[i]
        path = os.path.join(self.folder, f"{row['attachment_id']}.npy")

        video = np.load(path)

        if self.transform:
            video = self.transform(video)
        else:
            video = torch.tensor(video, dtype=torch.float32).permute(0,3,1,2) / 255.0

        label = self.label_map[row["text"]]
        return video, label

In [30]:
from torch.utils.data import DataLoader

train_ds = GestureDataset(train_df, os.path.join(CACHE_DIR, "train"), augment_video)
val_ds = GestureDataset(val_df, os.path.join(CACHE_DIR, "val"))
test_ds = GestureDataset(test_df, os.path.join(CACHE_DIR, "test"))

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=4)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=4)
test_loader = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=4)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [31]:
import torch.nn as nn
import torchvision.models as models

class VideoModel(nn.Module):
    def __init__(self, n_classes=10, dropout=0.5):
        super().__init__()

        backbone = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
        self.features = backbone.features

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(1280, n_classes)
        )

    def forward(self, x):
        b, t, c, h, w = x.shape

        x = x.view(b * t, c, h, w)
        x = self.features(x)
        x = self.pool(x)

        x = x.view(b, t, -1)
        x = x.mean(dim=1)

        return self.classifier(x)

In [32]:
import torch.optim as optim

device = "cuda" if torch.cuda.is_available() else "cpu"

model = VideoModel().to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.2)
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=4e-5)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

best_acc = 0

for epoch in range(9):
    model.train()
    total, correct = 0, 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)

        loss.backward()
        optimizer.step()

        pred = out.argmax(1)
        correct += (pred == y).sum().item()
        total += y.size(0)

    train_acc = correct / total

    model.eval()
    total, correct = 0, 0

    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            out = model(x)

            pred = out.argmax(1)
            correct += (pred == y).sum().item()
            total += y.size(0)

    val_acc = correct / total

    print(f"{epoch+1}: train={train_acc:.3f} val={val_acc:.3f}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "best.pth")

    scheduler.step()

Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 180MB/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


1: train=0.127 val=0.194
2: train=0.286 val=0.237
3: train=0.554 val=0.602
4: train=0.746 val=0.656
5: train=0.833 val=0.667
6: train=0.885 val=0.656
7: train=0.910 val=0.677
8: train=0.934 val=0.720
9: train=0.957 val=0.720


In [33]:
model.load_state_dict(torch.load("best.pth"))
model.eval()

correct, total = 0, 0

with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)

        out = model(x)
        pred = out.argmax(1)

        correct += (pred == y).sum().item()
        total += y.size(0)

print("Test accuracy:", correct / total)

Test accuracy: 0.645


In [37]:
import random

param_grid = {
    "freeze": [0, 5, 10, 15, 19],
    "drop": [0.3, 0.4, 0.5, 0.6],
    "smooth": [0.0, 0.1, 0.2, 0.3]
}

TRIALS = 3
EPOCHS_PER_RUN = 9

In [38]:
def freeze_backbone_layers(model, freeze_until):
    for name, param in model.features.named_parameters():
        try:
            layer_id = int(name.split('.')[0])
            param.requires_grad = layer_id >= freeze_until
        except:
            param.requires_grad = True

In [39]:
best_score = 0
best_params = None
best_path = "best_rs_model.pth"

print("Start Random Search")

for i in range(TRIALS):
    cfg = {k: random.choice(v) for k, v in param_grid.items()}

    print(f"\nTrial {i+1}/{TRIALS} -> {cfg}")

    model = VideoModel(n_classes=len(train_ds.labels), dropout=cfg["drop"]).to(device)

    freeze_backbone_layers(model, cfg["freeze"])

    loss_fn = nn.CrossEntropyLoss(label_smoothing=cfg["smooth"])

    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=1e-4,
        weight_decay=4e-5
    )

    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=EPOCHS_PER_RUN
    )

    local_best = 0

    for ep in range(EPOCHS_PER_RUN):
        # train
        model.train()
        total, correct = 0, 0

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()
            out = model(x)
            loss = loss_fn(out, y)

            loss.backward()
            optimizer.step()

            pred = out.argmax(1)
            correct += (pred == y).sum().item()
            total += y.size(0)

        train_acc = correct / total

        # val
        model.eval()
        total, correct = 0, 0

        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                out = model(x)

                pred = out.argmax(1)
                correct += (pred == y).sum().item()
                total += y.size(0)

        val_acc = correct / total

        print(f"  ep {ep+1}: train={train_acc:.3f} val={val_acc:.3f}")

        local_best = max(local_best, val_acc)
        scheduler.step()

    print("  best val:", local_best)

    if local_best > best_score:
        best_score = local_best
        best_params = cfg

        torch.save({
            "model": model.state_dict(),
            "params": cfg,
            "score": best_score
        }, best_path)

        print("  >>> new best <<<")

print("\nBest config:", best_params)
print("Best val acc:", best_score)

Start Random Search

Trial 1/3 -> {'freeze': 15, 'drop': 0.5, 'smooth': 0.0}
  ep 1: train=0.114 val=0.097
  ep 2: train=0.232 val=0.161
  ep 3: train=0.345 val=0.215
  ep 4: train=0.504 val=0.269
  ep 5: train=0.566 val=0.409
  ep 6: train=0.633 val=0.441
  ep 7: train=0.707 val=0.462
  ep 8: train=0.728 val=0.473
  ep 9: train=0.714 val=0.441
  best val: 0.4731182795698925
  >>> new best <<<

Trial 2/3 -> {'freeze': 0, 'drop': 0.5, 'smooth': 0.0}
  ep 1: train=0.124 val=0.108
  ep 2: train=0.342 val=0.269
  ep 3: train=0.578 val=0.419
  ep 4: train=0.713 val=0.602
  ep 5: train=0.779 val=0.602
  ep 6: train=0.848 val=0.613
  ep 7: train=0.852 val=0.645
  ep 8: train=0.892 val=0.624
  ep 9: train=0.884 val=0.645
  best val: 0.6451612903225806
  >>> new best <<<

Trial 3/3 -> {'freeze': 10, 'drop': 0.5, 'smooth': 0.3}
  ep 1: train=0.119 val=0.183
  ep 2: train=0.295 val=0.301
  ep 3: train=0.545 val=0.473
  ep 4: train=0.667 val=0.602
  ep 5: train=0.771 val=0.677
  ep 6: train=0.833 

In [40]:
checkpoint = torch.load("best_rs_model.pth", map_location=device)

model = VideoModel(
    n_classes=len(test_ds.labels),
    dropout=checkpoint["params"]["drop"]
).to(device)

model.load_state_dict(checkpoint["model"])
model.eval()

correct, total = 0, 0

with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)

        out = model(x)
        pred = out.argmax(1)

        correct += (pred == y).sum().item()
        total += y.size(0)

print("RS Test accuracy:", correct / total)

RS Test accuracy: 0.59


In [41]:
def mc_dropout_eval(model_path, passes=30):
    ckpt = torch.load(model_path, map_location=device)

    model = VideoModel(
        n_classes=len(test_ds.labels),
        dropout=ckpt["params"]["drop"]
    ).to(device)

    model.load_state_dict(ckpt["model"])

    # ВАЖНО: включаем dropout
    model.train()

    all_means = []
    all_stds = []
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(device)

            preds_mc = []

            for _ in range(passes):
                out = model(x)
                probs = torch.softmax(out, dim=1)
                preds_mc.append(probs)

            preds_mc = torch.stack(preds_mc)

            mean = preds_mc.mean(dim=0)
            std = preds_mc.std(dim=0)

            pred = mean.argmax(1)

            all_means.append(mean.cpu())
            all_stds.append(std.cpu())
            all_labels.append(y)
            all_preds.append(pred.cpu())

    means = torch.cat(all_means)
    stds = torch.cat(all_stds)
    labels = torch.cat(all_labels)
    preds = torch.cat(all_preds)

    acc = (preds == labels).float().mean().item()
    uncertainty = stds.mean().item()

    print("MC accuracy:", acc)
    print("Uncertainty:", uncertainty)

    return acc, uncertainty

In [42]:
mc_dropout_eval("best_rs_model.pth", passes=30)

MC accuracy: 0.25999999046325684
Uncertainty: 0.017633523792028427


(0.25999999046325684, 0.017633523792028427)

По итогу из-за сложности в вычислительных ресурсах не получилось улучшить бейзлайн с помощью методов Монте-Карло. Я могу это связать с тем, что у нас в принципе мало данных и из-за этого не получается улучшить бейзлайн.